# Iteration 2 — Multimodal TS Forecasting

**Tracks:**
- Track 1: Text pipeline ablation (D-series)
- Track 2: Data fraction sweep
- Track 3: Diagnostic instrumentation

**Runtime:** Colab T4  
**Storage:** Google Drive at `/content/drive/MyDrive/multimodal_TS_research/iteration_2/`

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/multimodal_TS_research/iteration_2'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
EMBEDDING_DIR  = f'{DRIVE_BASE}/embeddings'
REGISTRY_PATH  = f'{DRIVE_BASE}/iteration2_registry.json'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DIR,  exist_ok=True)
print('Drive mounted. Directories ready.')

In [ ]:
# Clone repo if not already present, otherwise pull latest
import subprocess, os

REPO_DIR = '/content/multimodal_TS_research'
REPO_URL = 'https://github.com/YOUR_USERNAME/multimodal_TS_research.git'  # update as needed

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Repo ready:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Symlink datasets from Drive (saves Colab re-download time)
DRIVE_DATASETS = '/content/drive/MyDrive/multimodal_TS_research/datasets'
LOCAL_DATASETS = f'{REPO_DIR}/dataset'

if os.path.exists(DRIVE_DATASETS) and not os.path.islink(LOCAL_DATASETS):
    if os.path.exists(LOCAL_DATASETS):
        os.rename(LOCAL_DATASETS, LOCAL_DATASETS + '_bak')
    os.symlink(DRIVE_DATASETS, LOCAL_DATASETS)
    print('Symlinked datasets from Drive')
else:
    print('Using local dataset directory:', LOCAL_DATASETS)

## 1. Registry helpers

In [ ]:
import json, shutil
from datetime import datetime
from pathlib import Path


def load_registry():
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return []


def save_registry(registry):
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(registry, f, indent=2)


def is_already_done(exp_name: str) -> bool:
    """Skip if this exact experiment name already has an entry."""
    registry = load_registry()
    return any(r['name'] == exp_name for r in registry)


def register_run(config: dict, metrics: dict, local_result_dir: str):
    """Copy checkpoint to Drive, append entry to iteration2_registry.json."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_name  = config.get('name', 'experiment')

    local_ckpt = os.path.join(local_result_dir, 'checkpoint.pth')
    drive_ckpt = os.path.join(CHECKPOINT_DIR, f'{exp_name}_{timestamp}.pth')

    entry = {
        'name':           exp_name,
        'model':          config['model']['name'],
        'dataset':        config['data']['dataset'],
        'train_fraction': config['training'].get('train_fraction', 1.0),
        'pred_len':       config['model']['pred_len'],
        'seed':           config['training'].get('seed', 2024),
        'text_source':    config['model'].get('text_source', None),
        'metrics':        {k: v for k, v in metrics.items() if k != 'diagnostics'},
        'diagnostics':    metrics.get('diagnostics', {}),
        # Write expected Drive path now; copy happens below.
        # If Colab crashes during copy, this entry already marks the run as done.
        'checkpoint_path': drive_ckpt if os.path.exists(local_ckpt) else None,
        'timestamp':      timestamp,
    }

    # Write to Drive registry BEFORE the slow checkpoint copy.
    # This ensures is_already_done() returns True even if copy is interrupted.
    registry = load_registry()
    registry.append(entry)
    save_registry(registry)

    # Copy checkpoint (may be slow; registry is already committed above)
    if os.path.exists(local_ckpt):
        shutil.copy2(local_ckpt, drive_ckpt)

    print(f'  Registered: {exp_name}  MAE={metrics["mae"]:.4f}  MSE={metrics["mse"]:.4f}')
    return entry


print(f'Registry has {len(load_registry())} entries.')

## 2. Run helper

In [ ]:
import yaml
import sys
sys.path.insert(0, REPO_DIR)

from run_experiment import run, load_config, set_seed, apply_overrides


def run_iter2(config_path: str, skip_if_done: bool = True, overrides: list = None,
              run_idx: int = None, total: int = None):
    """
    Run a single Iter 2 experiment.
    - Skips if experiment name already in registry (resume-safe).
    - Copies checkpoint to Drive and appends to iteration2_registry.json.
    - run_idx / total: used only for progress display (e.g. 20/256).
    """
    config = load_config(config_path)
    if overrides:
        config = apply_overrides(config, overrides)

    exp_name = config.get('name', 'experiment')
    counter  = f'[{run_idx}/{total}]  ' if run_idx is not None else ''

    if skip_if_done and is_already_done(exp_name):
        print(f'{counter}SKIP (already done): {exp_name}')
        return None

    print(f'\n{"=" * 70}')
    print(f'{counter}{exp_name}')
    print(f'{"=" * 70}')

    # run() writes to experiments/results/ locally
    metrics = run(config_path, overrides=overrides)

    # Find the result_dir that was just created (most recent under experiments/results/)
    results_root = Path(REPO_DIR) / 'experiments' / 'results'
    candidates = sorted(results_root.glob(f'{exp_name}_*'), key=lambda p: p.stat().st_mtime)
    local_result_dir = str(candidates[-1]) if candidates else str(results_root)

    return register_run(config, metrics, local_result_dir)


print('Run helper ready.')

## 3. Offline description encoding

Run once per dataset/split before any offline-mode experiments.

In [ ]:
import subprocess

DATASETS_TO_ENCODE = ['ETTh1', 'ETTh2', 'ETTm1']

for ds in DATASETS_TO_ENCODE:
    root = './dataset/ETT-small/'
    csv  = f'{ds}.csv'
    freq = 'h' if 'h' in ds.lower() else 't'
    for split in ['train', 'val', 'test']:
        out_path = f'{EMBEDDING_DIR}/{ds}_{split}_minilm.npy'
        if os.path.exists(out_path):
            print(f'Already encoded: {out_path}')
            continue
        result = subprocess.run(
            [
                'python', f'{REPO_DIR}/utils/text/encode_descriptions.py',
                '--dataset', ds, '--split', split,
                '--root_path', root, '--data_path', csv, '--freq', freq,
                '--out', out_path,
            ],
            cwd=REPO_DIR,
        )
        if result.returncode != 0:
            raise RuntimeError(f'Encoding failed for {ds}/{split} (exit {result.returncode})')
        print(f'Saved: {out_path}')

## 4. Generate experiment configs

In [ ]:
# D-series: 72 configs
!python utils/experiment/generate_configs.py --track d_series

# Tier 1: full fraction sweep
!python utils/experiment/generate_configs.py --track tier1

## 5. Run experiments

In [ ]:
import glob

# ── Build ordered run queue ────────────────────────────────────────────────────
# Priority ordering: D-series → Tier1 priority (PatchTST+EnsembleFusion/ETTh1)
#                   → remaining Tier1 (all models, ETTh1 → ETTh2 → ETTm1)

PRIORITY_MODELS   = ['PatchTST', 'EnsembleFusion']
PRIORITY_DATASETS = ['etth1']

def should_run_first(cfg_path: str) -> bool:
    p = os.path.basename(cfg_path).lower()
    return any(m.lower() in p for m in PRIORITY_MODELS) and any(d in p for d in PRIORITY_DATASETS)

d_configs  = sorted(glob.glob('experiments/configs/d_*.yaml'))
t1_configs = sorted(glob.glob('experiments/configs/t1_*.yaml'))

t1_priority = [c for c in t1_configs if should_run_first(c)]
t1_rest     = [c for c in t1_configs if not should_run_first(c)]

run_queue   = d_configs + t1_priority + t1_rest
TOTAL_RUNS  = len(run_queue)

print(f'Run queue: {TOTAL_RUNS} configs total')
print(f'  D-series       : {len(d_configs)}')
print(f'  Tier1 priority : {len(t1_priority)}')
print(f'  Tier1 rest     : {len(t1_rest)}')

In [ ]:
# Re-run this cell after a Colab disconnect — already-done experiments are skipped automatically.

already_done = sum(1 for cfg in run_queue if is_already_done(
    (lambda c: (load_config(c) or {}).get('name', ''))(cfg)
))
print(f'Queue: {TOTAL_RUNS} total  |  {already_done} already done  |  {TOTAL_RUNS - already_done} remaining\n')

for run_idx, cfg_path in enumerate(run_queue, start=1):
    run_iter2(cfg_path, run_idx=run_idx, total=TOTAL_RUNS)

## 6. Results summary

In [ ]:
import pandas as pd

registry = load_registry()
rows = []
for r in registry:
    rows.append({
        'name':     r['name'],
        'model':    r['model'],
        'dataset':  r['dataset'],
        'fraction': r.get('train_fraction', 1.0),
        'pred_len': r['pred_len'],
        'seed':     r.get('seed'),
        'text_src': r.get('text_source', '-'),
        'MAE':      r['metrics'].get('mae'),
        'MSE':      r['metrics'].get('mse'),
    })

df = pd.DataFrame(rows)
print(f'Total runs: {len(df)}')
df.sort_values(['dataset', 'model', 'fraction', 'pred_len']).head(40)

In [ ]:
# Mean ± std across seeds
summary = (
    df.groupby(['model', 'dataset', 'fraction', 'pred_len'])[['MAE', 'MSE']]
    .agg(['mean', 'std'])
    .round(4)
)
summary